In [1]:
import os
import numpy as np
import pandas as pd

# Part 1: Data Cleaning

# Load the CSV data dynamically from current directory
file_name = "Lab 4 - Retail_Sales_Data.csv"
current_dir = os.getcwd()
file_path = os.path.join(current_dir, file_name)

# 1. Loading the Data
df = pd.read_csv(file_path)
print("First five rows of the dataset:")
print(df.head())

# 2. Handling Missing Values
print("Missing values in columns:")
print(df.isnull().sum())

# Fill missing numerics with column mean
for col in ['Quantity', 'Price', 'Total_Spent', 'Discount']:
    df[col].fillna(df[col].mean(), inplace=True)
# Drop any row left with missing values (e.g., Payment_Method if still missing)
df.dropna(inplace=True)
print("Missing values after cleaning:")
print(df.isnull().sum())

# 3. Data Type Conversion
for col in ['Quantity', 'Price', 'Total_Spent', 'Discount']:
    df[col] = pd.to_numeric(df[col])
df['Date'] = pd.to_datetime(df['Date'])
print("Data types after conversion:")
print(df.dtypes)

# Part 2: Data Transformation

# 4. Create the Total_Cost column
df['Total_Cost'] = df['Quantity'] * df['Price'] - df['Discount']

# 5. Feature Engineering
df['Day_of_Week'] = df['Date'].dt.day_name()
df['Is_Discounted'] = df['Discount'] > 0

# 6. Handling Outliers (using IQR)
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outlier_mask = (df[column] < lower) | (df[column] > upper)
    print(f"Removed {outlier_mask.sum()} outliers from column: {column}")
    return df.loc[~outlier_mask]

df = remove_outliers(df, 'Total_Spent')
df = remove_outliers(df, 'Quantity')

# Part 3: Data Analysis

# 7. Descriptive Statistics
print("\nDescriptive statistics for numeric columns:")
print(df[['Quantity', 'Price', 'Total_Spent', 'Discount', 'Total_Cost']].describe())

# 8. Transaction Analysis
top_customer = df.groupby('Customer_ID')['Total_Spent'].sum().idxmax()
print(f"\nCustomer with highest total spending: {top_customer}")

top_product = df.groupby('Product')['Quantity'].sum().idxmax()
print(f"Most popular product (most purchased): {top_product}")

# 9. Time-Based Analysis
sales_by_day = df.groupby('Day_of_Week')['Total_Cost'].sum()
print("\nTotal sales by day of week:")
print(sales_by_day)

print(f"Day with highest total sales: {sales_by_day.idxmax()}")
print(f"Day with lowest total sales: {sales_by_day.idxmin()}")

# 10. Correlation Analysis
print("\nCorrelation between Total_Spent, Price, and Quantity:")
print(df[['Total_Spent', 'Price', 'Quantity']].corr())

# Done: the notebook is now organized per your objectives, using original variable names and columns.


First five rows of the dataset:
   Transaction_ID                 Date  Customer_ID     Product  Quantity  \
0            6119  2023-08-02 06:00:00         2950  Headphones         1   
1            9932  2024-01-08 03:00:00         2272  Headphones         3   
2            9516  2023-12-21 19:00:00         2007      Tablet         1   
3            8283  2023-10-31 10:00:00         2787    Keyboard         3   
4            8624  2023-11-14 15:00:00         2281       Mouse         2   

     Price  Total_Spent  Discount Payment_Method  
0      NaN       738.09     85.18         PayPal  
1  1652.05      4925.71     30.44    Credit Card  
2   660.58       643.71     16.87           Cash  
3  1008.84      2957.98     68.54         PayPal  
4  1091.05      2154.41     27.69     Debit Card  
Missing values in columns:
Transaction_ID      0
Date                0
Customer_ID         0
Product             0
Quantity            0
Price             520
Total_Spent       316
Discount          

/var/folders/l6/4gxtgw5j6c99mmhtcx061w0h0000gn/T/ipykernel_49605/1876660613.py:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mean(), inplace=True)
